In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [8]:
%%writefile part1_bitonic_sort.cu

// part1_bitonic_sort.cu
#include <iostream>      // Библиотека для ввода-вывода: используется для вывода информации о выполнении программы в консоль (Лекция №1: отладка параллельных программ)
#include <vector>        // Динамический контейнер vector на CPU (хосте) — удобен для хранения больших массивов данных перед передачей на GPU (Лекция №3: различие хост- и device-памяти)
#include <algorithm>     // Для функции std::generate (заполнение массива) и std::is_sorted (проверка корректности сортировки) — стандартные алгоритмы C++
#include <random>        // Для генерации псевдослучайных чисел: mt19937 и uniform_int_distribution — качественный ГСЧ для тестовых данных
#include <chrono>        // Для высокоточного измерения времени выполнения на GPU — важно для оценки производительности параллельных алгоритмов (Лекция №1: сравнение CPU и GPU)
#include <cuda_runtime.h> // Основной заголовок CUDA: содержит функции управления памятью, запуска ядер и проверки ошибок (Лекция №3: базовые API CUDA)

using namespace std;     // Упрощает код: позволяет использовать cout, vector, chrono без префикса std:: — стандартная практика в учебных программах

// Макрос для проверки ошибок CUDA: если операция не удалась — выводит сообщение и завершает программу (Лекция №3: важность обработки ошибок в CUDA)
#define CUDA_CHECK(err) do { \
    cudaError_t local_err = (err); \
    if (local_err != cudaSuccess) { \
        cerr << "CUDA error: " << cudaGetErrorString(local_err) << " at line " << __LINE__ << endl; \
        exit(1); \
    } \
} while(0)

// Ядро CUDA: выполняется параллельно тысячами потоков на GPU (Лекция №3: __global__ функция — точка входа с хоста на устройство)
__global__ void bitonic_sort_step(int *dev_array, int j, int k) {
    // Вычисляем глобальный индекс текущего потока: threadIdx.x — внутри блока, blockDim.x — размер блока, blockIdx.x — номер блока (Лекция №3: индексация потоков в CUDA)
    unsigned int i = threadIdx.x + blockDim.x * blockIdx.x;

    // Вычисляем индекс парного элемента для сравнения с помощью побитовой операции XOR — ключевой приём в битонной сортировке
    unsigned int ixj = i ^ j;

    // Проверяем границы: парный индекс должен быть больше текущего и не выходить за пределы массива (избежание ошибок доступа к памяти)
    if (ixj > i) {
        // Определяем направление сравнения по значению бита k (Лекция №3: битонная последовательность чередует восходящие и нисходящие фазы)
        if ((i & k) == 0) {
            // Восходящая фаза: меньший элемент должен оказаться по меньшему индексу
            if (dev_array[i] > dev_array[ixj]) {
                // Обмен элементов местами — атомарная операция внутри потока
                int temp = dev_array[i];
                dev_array[i] = dev_array[ixj];
                dev_array[ixj] = temp;
            }
        } else {
            // Нисходящая фаза: больший элемент должен оказаться по меньшему индексу
            if (dev_array[i] < dev_array[ixj]) {
                int temp = dev_array[i];
                dev_array[i] = dev_array[ixj];
                dev_array[ixj] = temp;
            }
        }
    }
    // Каждый поток независимо обрабатывает свою пару — демонстрирует массовую параллельность GPU (Лекция №3: тысячи потоков выполняют простые операции одновременно)
}

int main() {  // Главная функция на CPU (хосте) — управляет всей программой (Лекция №3: гетерогенная модель: CPU управляет, GPU вычисляет)
    const int N = 1 << 16;  // Размер массива — 65536 элементов. Обязательно степень двойки для корректной работы битонной сортировки (алгоритмическое требование)
    vector<int> host_array(N);  // Массив на CPU: здесь хранятся исходные и конечные данные (Лекция №3: хост-память медленнее, но доступна для ввода-вывода)

    cout << "Part 1: Bitonic Merge Sort on GPU (CUDA)\n";  // Заголовок части — соответствует структуре практических работ
    cout << "Initializing array with " << N << " random elements...\n";  // Сообщение о начале инициализации

    // Генератор случайных чисел для создания тестовых данных
    mt19937 gen(time(nullptr));  // Инициализация seed текущим временем — обеспечивает разные данные при каждом запуске
    uniform_int_distribution<int> dist(1, 100000);  // Диапазон значений от 1 до 100000 — создаёт разнообразные данные
    generate(host_array.begin(), host_array.end(), [&]() { return dist(gen); });  // Заполнение массива случайными числами

    cout << "First 10 elements (before sorting): ";  // Вывод первых элементов для визуальной проверки исходного состояния
    for (int i = 0; i < 10 && i < N; ++i) cout << host_array[i] << " ";
    cout << "...\n";

    cout << "Last 10 elements (before sorting): ";  // Вывод последних элементов — помогает увидеть хаотичность данных
    for (int i = max(0, N - 10); i < N; ++i) cout << host_array[i] << " ";
    cout << "\n\n";

    int *dev_array = nullptr;  // Указатель на массив в глобальной памяти GPU (device)

    cout << "Allocating memory on GPU (" << N * sizeof(int) / 1024 / 1024 << " MB)...\n";  // Сообщение о выделении памяти
    CUDA_CHECK(cudaMalloc(&dev_array, N * sizeof(int)));  // Выделение памяти на GPU — критически важный шаг (Лекция №3: явное управление памятью)

    cout << "Copying data from CPU to GPU...\n";  // Копирование данных на устройство
    CUDA_CHECK(cudaMemcpy(dev_array, host_array.data(), N * sizeof(int), cudaMemcpyHostToDevice));  // Передача данных — bottleneck в гетерогенных программах (Лекция №1)

    // Конфигурация запуска ядер CUDA
    dim3 threads(256);  // 256 потоков на блок — хороший баланс для большинства GPU (Лекция №3: выбор размера блока влияет на occupancy)
    dim3 blocks((N + threads.x - 1) / threads.x);  // Вычисляем необходимое количество блоков с округлением вверх

    cout << "Launch configuration: " << blocks.x << " blocks × " << threads.x << " threads = "
         << blocks.x * threads.x << " total threads\n\n";  // Вывод конфигурации — демонстрирует масштабируемость

    cout << "Starting GPU sorting (bitonic phases)...\n";  // Начало основного вычисления на GPU
    auto start_gpu = chrono::high_resolution_clock::now();  // Замер времени начала выполнения на GPU

    int phase_count = 0;  // Счётчик выполненных фаз для отладки
    // Основной цикл битонной сортировки
    for (int k = 2; k <= N; k <<= 1) {  // k — размер битонной последовательности: 2, 4, 8, ..., N (логарифмическое количество уровней)
        for (int j = k >> 1; j > 0; j >>= 1) {  // j — расстояние между сравниваемыми элементами в текущей последовательности
            // Запуск ядра на GPU с заданной конфигурацией
            bitonic_sort_step<<<blocks, threads>>>(dev_array, j, k);
            CUDA_CHECK(cudaGetLastError());  // Проверка ошибок запуска ядра
            CUDA_CHECK(cudaDeviceSynchronize());  // Синхронизация: ждём завершения всех потоков перед следующей фазой (Лекция №3: необходима для корректности)
            phase_count++;  // Увеличиваем счётчик фаз

            // Вывод прогресса каждые 8 фаз — чтобы консоль не засорялась, но было видно движение
            if (phase_count % 8 == 0 || j == 1) {
                cout << "  Completed phase " << phase_count << " (k=" << k << ", j=" << j << ")\n";
            }
        }
    }

    auto end_gpu = chrono::high_resolution_clock::now();  // Замер времени окончания
    chrono::duration<double, milli> gpu_time_ms = end_gpu - start_gpu;  // Время в миллисекундах для точности

    cout << "\nGPU sorting completed in " << gpu_time_ms.count() << " ms\n";  // Итоговое время выполнения на GPU

    cout << "Copying sorted data back to CPU...\n";  // Копирование результата обратно
    CUDA_CHECK(cudaMemcpy(host_array.data(), dev_array, N * sizeof(int), cudaMemcpyDeviceToHost));

    cout << "Freeing GPU memory...\n";  // Освобождение памяти — обязательная практика (Лекция №3: избежание утечек)
    CUDA_CHECK(cudaFree(dev_array));

    // Проверка корректности результата
    bool is_sorted = std::is_sorted(host_array.begin(), host_array.end());  // Используем стандартный алгоритм для верификации

    // Вывод отсортированных элементов для визуального подтверждения
    cout << "\nFirst 10 elements (after sorting): ";
    for (int i = 0; i < 10 && i < N; ++i) cout << host_array[i] << " ";
    cout << "...\n";

    cout << "Last 10 elements (after sorting): ";
    for (int i = max(0, N - 10); i < N; ++i) cout << host_array[i] << " ";
    cout << "\n\n";

    // Финальный отчёт
    cout << "Sorting correct: " << (is_sorted ? "Yes" : "No") << "\n";
    cout << "Total GPU execution time: " << gpu_time_ms.count() / 1000.0 << " seconds\n";
    cout << "Total phases executed: " << phase_count << "\n";
    cout << "Note: Bitonic sort is a parallel version of merge sort suitable for GPU (Lecture #3)\n";

    return 0;  // Успешное завершение программы
}

Overwriting part1_bitonic_sort.cu


In [9]:
!nvcc -std=c++17 part1_bitonic_sort.cu -o bitonic_sort
!./bitonic_sort

Part 1: Bitonic Merge Sort on GPU (CUDA)
Initializing array with 65536 random elements...
First 10 elements (before sorting): 39261 97950 95666 66926 23337 92720 11843 34274 26040 65488 ...
Last 10 elements (before sorting): 92386 85742 68024 96637 74467 58249 23175 28242 61150 9422 

Allocating memory on GPU (0 MB)...
Copying data from CPU to GPU...
Launch configuration: 256 blocks × 256 threads = 65536 total threads

Starting GPU sorting (bitonic phases)...
CUDA error: the provided PTX was compiled with an unsupported toolchain. at line 96


In [2]:
%%writefile part2_quick_sort_hybrid.cu

// part2_quick_sort_hybrid.cu — оптимизированная гибридная Quick Sort для GPU (быстрая работа в Colab)
#include <iostream>      // Библиотека для ввода-вывода: используется для вывода результатов и отладочной информации (Лекция №1: отладка гетерогенных программ)
#include <vector>        // Динамический контейнер vector на CPU (хосте) — хранение исходного и отсортированного массива (Лекция №3: хост-память для ввода-вывода)
#include <algorithm>     // Для std::swap (обмен элементов) и std::is_sorted (проверка корректности) — стандартные алгоритмы C++
#include <random>        // Для генерации случайных чисел: mt19937 и uniform_int_distribution — качественный ГСЧ для тестовых данных
#include <chrono>        // Для высокоточного измерения времени выполнения всей сортировки — сравнение CPU и GPU (Лекция №1: оценка производительности)
#include <cuda_runtime.h> // Основной заголовок CUDA: функции управления памятью, запуска ядер и синхронизации (Лекция №3: базовый API CUDA)

using namespace std;     // Упрощает код: позволяет использовать cout, vector, swap без префикса std:: — стандартная практика в учебных программах

// Макрос для проверки ошибок CUDA: при ошибке выводит сообщение с номером строки и завершает программу (Лекция №3: обязательная обработка ошибок в CUDA)
#define CUDA_CHECK(err) do { \
    if (err != cudaSuccess) { \
        cerr << "CUDA error: " << cudaGetErrorString(err) << " at line " << __LINE__ << endl; \
        exit(1); \
    } \
} while(0)

// Ядро CUDA: каждый поток проверяет, меньше ли его элемент опорного, и атомарно увеличивает счётчик (Лекция №3: атомарные операции для безопасного доступа из тысяч потоков)
__global__ void count_less_than(int *data, int pivot, int *count, int n) {
    // Вычисляем глобальный индекс текущего потока в сетке (grid) — стандартная индексация в CUDA (Лекция №3: threadIdx, blockIdx, blockDim)
    int idx = threadIdx.x + blockDim.x * blockIdx.x;

    // Проверяем, что индекс не выходит за границы массива — избежание ошибок чтения памяти
    if (idx < n) {
        // Если элемент меньше опорного — атомарно увеличиваем глобальный счётчик
        if (data[idx] < pivot) {
            atomicAdd(count, 1);  // Атомарная операция — безопасна для параллельного выполнения тысячами потоков (Лекция №3: разрешение гонок данных)
        }
    }
    // Каждый поток независимо проверяет свой элемент — демонстрирует массовую параллельность GPU (Лекция №3: простые операции на тысячах ядер)
}

// Рекурсивная функция Quick Sort: использует GPU только для подсчёта элементов меньше pivot (оптимизированный гибридный подход)
void quick_sort_hybrid(vector<int>& arr, int low, int high) {
    // Базовый случай рекурсии: если подмассив имеет 1 или 0 элементов — сортировка не нужна
    if (low >= high) return;

    // Выбираем опорный элемент (pivot) — последний элемент подмассива (стандартный выбор в Quick Sort)
    int pivot = arr[high];

    // Размер текущего подмассива
    int n = high - low + 1;

    // Указатели на массивы в глобальной памяти GPU
    int *d_data = nullptr;   // Для копии подмассива
    int *d_count = nullptr;  // Для счётчика элементов меньше pivot

    // Выделяем память на GPU для данных и счётчика (Лекция №3: cudaMalloc — явное управление device-памятью)
    CUDA_CHECK(cudaMalloc(&d_data, n * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));

    // Копируем подмассив с CPU на GPU — bottleneck в гетерогенных программах (Лекция №1: передача данных между устройствами)
    CUDA_CHECK(cudaMemcpy(d_data, &arr[low], n * sizeof(int), cudaMemcpyHostToDevice));

    // Инициализируем счётчик нулём на GPU
    int zero = 0;
    CUDA_CHECK(cudaMemcpy(d_count, &zero, sizeof(int), cudaMemcpyHostToDevice));

    // Настраиваем конфигурацию запуска ядра: 256 потоков на блок — хороший баланс для большинства GPU (Лекция №3: выбор размера блока влияет на occupancy)
    dim3 threads(256);
    dim3 blocks((n + threads.x - 1) / threads.x);  // Округляем количество блоков вверх

    // Запускаем ядро: каждый поток проверяет свой элемент и обновляет счётчик
    count_less_than<<<blocks, threads>>>(d_data, pivot, d_count, n);

    // Ждём завершения ядра — синхронизация необходима перед чтением результата (Лекция №3: cudaDeviceSynchronize)
    CUDA_CHECK(cudaDeviceSynchronize());

    // Копируем счётчик с GPU на CPU
    int h_count = 0;
    CUDA_CHECK(cudaMemcpy(&h_count, d_count, sizeof(int), cudaMemcpyDeviceToHost));

    // Вычисляем позицию pivot: все элементы меньше pivot должны быть слева от неё
    int pi = low + h_count;

    // Ручная перестановка элементов на CPU по вычисленной позиции (быстро для одного уровня)
    int i = low - 1;
    for (int j = low; j < high; ++j) {
        if (arr[j] < pivot) {
            ++i;
            swap(arr[i], arr[j]);
        }
    }
    swap(arr[i + 1], arr[high]);  // Ставим pivot на правильное место

    // Освобождаем память на GPU — обязательная практика (Лекция №3: избежание утечек памяти)
    CUDA_CHECK(cudaFree(d_data));
    CUDA_CHECK(cudaFree(d_count));

    // Рекурсивно сортируем левую часть (меньше pivot) — на CPU
    quick_sort_hybrid(arr, low, pi - 1);

    // Рекурсивно сортируем правую часть (больше или равно pivot) — на CPU
    quick_sort_hybrid(arr, pi + 1, high);

    // Гибридный подход: GPU ускоряет подсчёт (параллельные сравнения), CPU — рекурсию и перестановку (Лекция №1: эффективное распределение задач между CPU и GPU)
}

int main() {  // Главная функция на CPU — управляет всей программой (Лекция №3: CPU отвечает за последовательные части и запуск GPU)
    const int N = 1000000;  // Размер массива — 1 миллион элементов: достаточно большой для демонстрации ускорения на GPU
    vector<int> arr(N);     // Массив на CPU для хранения исходных и отсортированных данных

    cout << "Part 2: Optimized Hybrid Quick Sort (GPU count + CPU partition)\n";  // Заголовок части
    cout << "Array size: " << N << " elements\n";  // Вывод размера массива
    cout << "Generating random data...\n";  // Сообщение о генерации данных

    // Генератор случайных чисел
    mt19937 gen(time(nullptr));  // Seed от текущего времени — разные данные при каждом запуске
    uniform_int_distribution<int> dist(1, 1000000);  // Диапазон значений
    generate(arr.begin(), arr.end(), [&]() { return dist(gen); });  // Заполнение массива

    cout << "Starting hybrid sorting...\n";  // Начало сортировки

    // Замер времени всей сортировки
    auto start = chrono::high_resolution_clock::now();

    // Запуск оптимизированной гибридной Quick Sort
    quick_sort_hybrid(arr, 0, N - 1);

    auto end = chrono::high_resolution_clock::now();
    chrono::duration<double> time = end - start;  // Общее время выполнения

    // Проверка корректности результата
    bool sorted = std::is_sorted(arr.begin(), arr.end());

    // Вывод результатов на английском
    cout << "\nSorting completed!\n";
    cout << "Execution time: " << time.count() << " seconds\n";
    cout << "Sorting correct: " << (sorted ? "Yes" : "No") << "\n";
    cout << "Note: Optimized hybrid approach uses GPU for parallel counting and CPU for recursion (Lecture #1, #3)\n";

    return 0;  // Успешное завершение программы
}

Writing part2_quick_sort_hybrid.cu


In [10]:
!nvcc -std=c++17 -arch=sm_75 part2_quick_sort_hybrid.cu -o quick_sort_hybrid
!./quick_sort_hybrid

Part 2: Optimized Hybrid Quick Sort (GPU count + CPU partition)
Array size: 1000000 elements
Generating random data...
Starting hybrid sorting...

Sorting completed!
Execution time: 106.811 seconds
Sorting correct: Yes
Note: Optimized hybrid approach uses GPU for parallel counting and CPU for recursion (Lecture #1, #3)


In [6]:
%%writefile part3_heap_sort.cu

// part3_heap_sort.cu — оптимизированная версия Heap Sort на GPU для быстрой работы в Colab
#include <iostream>      // Библиотека для ввода-вывода: используется для вывода результатов и отладочной информации (Лекция №1: отладка гетерогенных программ)
#include <vector>        // Динамический контейнер vector на CPU (хосте) — хранение исходного и отсортированного массива (Лекция №3: хост-память для ввода-вывода)
#include <algorithm>     // Для std::is_sorted — проверка корректности сортировки после выполнения на GPU
#include <random>        // Для генерации случайных чисел: mt19937 и uniform_int_distribution — качественный ГСЧ для тестовых данных
#include <chrono>        // Для высокоточного измерения времени выполнения всей сортировки — сравнение CPU и GPU (Лекция №1: оценка производительности)
#include <cuda_runtime.h> // Основной заголовок CUDA: функции управления памятью, запуска ядер и синхронизации (Лекция №3: базовый API CUDA)

using namespace std;     // Упрощает код: позволяет использовать cout, vector, swap без префикса std:: — стандартная практика в учебных программах

// Макрос для проверки ошибок CUDA: при ошибке выводит сообщение с номером строки и завершает программу (Лекция №3: обязательная обработка ошибок в CUDA)
#define CUDA_CHECK(err) do { \
    cudaError_t local_err = (err); \
    if (local_err != cudaSuccess) { \
        cerr << "CUDA error: " << cudaGetErrorString(local_err) << " at line " << __LINE__ << endl; \
        exit(1); \
    } \
} while(0)

// Ядро CUDA для параллельного heapify: каждый поток обрабатывает свой узел кучи (Лекция №3: массовая параллельность для независимых операций)
__global__ void heapify_kernel(int *arr, int n, int root) {
    // Вычисляем глобальный индекс текущего потока — стандартная индексация в CUDA (Лекция №3: threadIdx, blockIdx, blockDim)
    int idx = threadIdx.x + blockDim.x * blockIdx.x;

    // Обрабатываем только родительские узлы (индексы от root до n/2) — листья не нуждаются в heapify
    if (idx >= root && idx < n / 2) {
        // Текущий узел — кандидат на максимум
        int largest = idx;

        // Вычисляем индексы левого и правого потомков
        int left = 2 * idx + 1;
        int right = 2 * idx + 2;

        // Сравниваем с левым потомком: если он существует и больше текущего максимума — обновляем
        if (left < n && arr[left] > arr[largest]) {
            largest = left;
        }

        // Сравниваем с правым потомком: если он существует и больше текущего максимума — обновляем
        if (right < n && arr[right] > arr[largest]) {
            largest = right;
        }

        // Если максимум не в корне поддерева — меняем элементы местами
        if (largest != idx) {
            // Обмен значениями — атомарная операция внутри потока
            int temp = arr[idx];
            arr[idx] = arr[largest];
            arr[largest] = temp;
        }
    }
    // Каждый поток независимо выполняет heapify для своего поддерева — демонстрирует параллелизм на GPU (Лекция №3: тысячи потоков для независимых задач)
}

// Оптимизированная функция Heap Sort на GPU: память выделяется один раз, параллельное построение кучи
void heap_sort_gpu(vector<int>& arr) {
    // Размер массива
    int n = arr.size();

    // Указатель на массив в глобальной памяти GPU
    int *d_arr = nullptr;

    cout << "Part 3: Optimized Heap Sort on GPU (parallel heapify)\n";  // Заголовок части
    cout << "Array size: " << n << " elements\n";  // Вывод размера массива
    cout << "Allocating GPU memory once...\n";  // Выделение памяти один раз — оптимизация (избегание тысяч cudaMalloc)

    // Выделяем память на GPU один раз для всего массива (Лекция №3: минимизация накладных расходов на управление памятью)
    CUDA_CHECK(cudaMalloc(&d_arr, n * sizeof(int)));

    cout << "Copying data from CPU to GPU...\n";  // Копирование данных — bottleneck в гетерогенных программах (Лекция №1)
    CUDA_CHECK(cudaMemcpy(d_arr, arr.data(), n * sizeof(int), cudaMemcpyHostToDevice));

    // Конфигурация запуска ядер: 256 потоков на блок — хороший баланс для большинства GPU (Лекция №3: выбор размера блока)
    dim3 threads(256);
    dim3 blocks((n / 2 + threads.x - 1) / threads.x);  // Округляем количество блоков для покрытия всех родительских узлов

    cout << "Grid configuration: " << blocks.x << " blocks x " << threads.x << " threads\n\n";  // Вывод конфигурации запуска

    cout << "Building max-heap in parallel...\n";  // Начало построения кучи

    // Параллельное построение кучи: начинаем с последнего родительского узла и идём вверх
    for (int i = n / 2 - 1; i >= 0; --i) {
        // Запускаем ядро для heapify от узла i
        heapify_kernel<<<blocks, threads>>>(d_arr, n, i);
        CUDA_CHECK(cudaGetLastError());  // Проверка ошибок запуска ядра
        CUDA_CHECK(cudaDeviceSynchronize());  // Синхронизация — ждём завершения всех потоков (Лекция №3: необходима для корректности)
    }

    cout << "Max-heap built. Extracting elements sequentially with parallel restore...\n";  // Куча построена — начинаем извлечение

    // Последовательное извлечение максимума (корня) с параллельным восстановлением кучи
    for (int i = n - 1; i > 0; --i) {
        // Меняем корень (максимум) и последний элемент
        int root_val, last_val;
        CUDA_CHECK(cudaMemcpy(&root_val, d_arr, sizeof(int), cudaMemcpyDeviceToHost));  // Читаем корень
        CUDA_CHECK(cudaMemcpy(&last_val, d_arr + i, sizeof(int), cudaMemcpyDeviceToHost));  // Читаем последний элемент

        CUDA_CHECK(cudaMemcpy(d_arr, &last_val, sizeof(int), cudaMemcpyHostToDevice));  // Записываем последний в корень
        CUDA_CHECK(cudaMemcpy(d_arr + i, &root_val, sizeof(int), cudaMemcpyHostToDevice));  // Записываем корень на конец

        // Параллельно восстанавливаем свойство кучи для уменьшенного массива (размер i)
        heapify_kernel<<<blocks, threads>>>(d_arr, i, 0);
        CUDA_CHECK(cudaDeviceSynchronize());
    }

    cout << "Copying sorted array back to CPU...\n";  // Копирование результата
    CUDA_CHECK(cudaMemcpy(arr.data(), d_arr, n * sizeof(int), cudaMemcpyDeviceToHost));

    cout << "Freeing GPU memory...\n";  // Освобождение памяти — обязательная практика (Лекция №3: избежание утечек)
    CUDA_CHECK(cudaFree(d_arr));
}

// Главная функция на CPU — управляет всей программой (Лекция №3: CPU отвечает за последовательные части и запуск GPU)
int main() {
    const int N = 524288;  // Размер массива — 524288 элементов (степень 2, достаточно большой для демонстрации)
    vector<int> arr(N);     // Массив на CPU для хранения данных

    cout << "Generating random data...\n";  // Генерация тестовых данных

    // Генератор случайных чисел
    mt19937 gen(time(nullptr));  // Seed от текущего времени — разные данные при каждом запуске
    uniform_int_distribution<int> dist(1, 1000000);  // Диапазон значений
    generate(arr.begin(), arr.end(), [&]() { return dist(gen); });  // Заполнение массива

    cout << "Starting optimized Heap Sort on GPU...\n";  // Начало сортировки

    // Замер времени всей сортировки
    auto start = chrono::high_resolution_clock::now();

    // Запуск оптимизированной Heap Sort на GPU
    heap_sort_gpu(arr);

    auto end = chrono::high_resolution_clock::now();
    chrono::duration<double> time = end - start;  // Общее время выполнения

    // Проверка корректности результата
    bool sorted = std::is_sorted(arr.begin(), arr.end());

    // Вывод результатов на английском
    cout << "\nHeap Sort completed!\n";
    cout << "Execution time: " << time.count() << " seconds\n";
    cout << "Sorting correct: " << (sorted ? "Yes" : "No") << "\n";
    cout << "Note: Optimized version with single memory allocation and parallel heapify (Lecture #3)\n";

    return 0;  // Успешное завершение программы
}

Overwriting part3_heap_sort.cu


In [8]:
!nvcc -std=c++17 -arch=sm_75 part3_heap_sort.cu -o part3_heap
!./part3_heap

Generating random data...
Starting optimized Heap Sort on GPU...
Part 3: Optimized Heap Sort on GPU (parallel heapify)
Array size: 524288 elements
Allocating GPU memory once...
Copying data from CPU to GPU...
Grid configuration: 1024 blocks x 256 threads

Building max-heap in parallel...
Max-heap built. Extracting elements sequentially with parallel restore...
Copying sorted array back to CPU...
Freeing GPU memory...

Heap Sort completed!
Execution time: 22.8261 seconds
Sorting correct: No
Note: Optimized version with single memory allocation and parallel heapify (Lecture #3)
